# A8. 竞品价格追踪与动态定价 Notebook

> **配套模块**: [A8 定价策略](../paths/a-operators/a8-pricing-strategy.md)
>
> **功能**: 竞品价格变化分析 + 价格弹性估算 + 动态定价建议
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-skills/blob/main/notebooks/a8-price-tracker.ipynb)

In [ ]:
!pip install -q pandas numpy plotly

## 1. 输入价格数据

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

PRICE_CSV = Path('price-history.csv')
if not PRICE_CSV.is_file():
    raise FileNotFoundError('请上传真实价格历史并命名为 price-history.csv')

df = pd.read_csv(PRICE_CSV)
required = {'date', 'my_price', 'comp_a_price', 'comp_b_price', 'comp_c_price', 'my_daily_sales'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'价格 CSV 缺少必需列: {sorted(missing)}')
if len(df) < 2:
    raise ValueError('价格历史至少需要 2 行')
df['date'] = pd.to_datetime(df['date'], errors='raise')
numeric = sorted(required - {'date'})
for column in numeric:
    df[column] = pd.to_numeric(df[column], errors='raise')
if (df[numeric] < 0).any().any():
    raise ValueError('价格和销量不能为负数')
if df['date'].duplicated().any():
    raise ValueError('date 不能重复')
df = df.sort_values('date').reset_index(drop=True)
days = len(df)
print(f'Price tracking: {days} days')
print(f'My price: ${df["my_price"].iloc[-1]:.2f}')
for label, column in [('Competitor A', 'comp_a_price'), ('Competitor B', 'comp_b_price'), ('Competitor C', 'comp_c_price')]:
    print(f'{label}: ${df[column].iloc[0]:.2f} → ${df[column].iloc[-1]:.2f}')


## 2. 价格趋势可视化

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['date'], y=df['my_price'], name='My Price', line=dict(width=3, color='blue')))
fig.add_trace(go.Scatter(x=df['date'], y=df['comp_a_price'], name='Competitor A', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=df['date'], y=df['comp_b_price'], name='Competitor B', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=df['date'], y=df['comp_c_price'], name='Competitor C', line=dict(dash='dash')))
fig.update_layout(title='60-Day Price Tracking', yaxis_title='Price ($)', xaxis_title='Date')
fig.show()

# 价格差异分析
print('=== Price Gap Analysis (Latest) ===')
for comp, col in [('Comp A', 'comp_a_price'), ('Comp B', 'comp_b_price'), ('Comp C', 'comp_c_price')]:
    gap = df['my_price'].iloc[-1] - df[col].iloc[-1]
    pct = gap / df['my_price'].iloc[-1] * 100
    emoji = '🔴' if gap > 3 else ('🟡' if gap > 0 else '🟢')
    print(f'{emoji} vs {comp}: ${gap:+.2f} ({pct:+.1f}%)')

## 3. 价格变化检测

In [ ]:
print('=== Price Change Alerts ===')
for comp, col in [('Competitor A', 'comp_a_price'), ('Competitor B', 'comp_b_price'), ('Competitor C', 'comp_c_price')]:
    prices = df[col]
    # 检测显著价格变化（>5%）
    pct_changes = prices.pct_change()
    significant = df[abs(pct_changes) > 0.05]
    if len(significant) > 0:
        for _, row in significant.iterrows():
            change = pct_changes[row.name]
            direction = '📉 降价' if change < 0 else '📈 涨价'
            print(f'{direction} {comp} on {row["date"].strftime("%m/%d")}: ${prices[row.name-1]:.2f} → ${row[col]:.2f} ({change*100:+.1f}%)')
    else:
        print(f'✅ {comp}: No significant changes')


## 4. 价格-销量相关性

In [ ]:
# 分析竞品降价对我的销量的影响
df['price_gap_a'] = df['my_price'] - df['comp_a_price']
corr = df['price_gap_a'].corr(df['my_daily_sales'])
print(f'Price gap vs sales correlation: {corr:.3f}')
if corr < -0.3:
    print('⚠️ Strong negative correlation: when competitor A is cheaper, your sales drop')

fig = px.scatter(df, x='price_gap_a', y='my_daily_sales', trendline='ols',
                 title='Price Gap (Me - Comp A) vs My Daily Sales',
                 labels={'price_gap_a': 'Price Gap ($)', 'my_daily_sales': 'Daily Sales'})
fig.show()

# 定价建议
print('\n=== Pricing Recommendations ===')
avg_gap = df['price_gap_a'].iloc[-7:].mean()
if avg_gap > 5:
    print(f'🔴 You are ${avg_gap:.2f} above Comp A. Consider:')
    print(f'   Option 1: Match at ${df["comp_a_price"].iloc[-1]:.2f} (risk: margin compression)')
    print(f'   Option 2: Partial match at ${(df["my_price"].iloc[-1] + df["comp_a_price"].iloc[-1])/2:.2f}')
    print(f'   Option 3: Hold price, differentiate on value (brand/quality/service)')
elif avg_gap > 0:
    print(f'🟡 Slight premium of ${avg_gap:.2f}. Monitor but no immediate action needed.')
else:
    print(f'🟢 You are competitively priced. Focus on conversion optimization.')


## 5. 导出

In [ ]:
df.to_csv('price_tracking.csv', index=False)
print('Exported to price_tracking.csv')